# 01.5 - Trees & Binary Search Trees

## What is a tree?
A hierarchy of **nodes** connected by edges, with one **root** and **no cycles**.

```
            8          <- root (depth 0)
          /   \
         3     10      <- children of 8
        / \      \
       1   6      14   <- 1, 6, 14 ... leaves have no children
          / \
         4   7
```
**Vocabulary:** root, parent, child, leaf, subtree, **depth** (distance from root), **height** (longest path down), **binary tree** (≤ 2 children).

## Binary Search Tree (BST)
For every node: **left subtree < node < right subtree**. The tree above is a BST.
Search / insert / delete = **O(h)** → O(log n) if balanced, O(n) if it degenerates into a line.
Self-balancing versions (**AVL, Red-Black trees**) guarantee O(log n).

## Where are trees used?
| Real system | Tree |
|---|---|
| **File systems** (folders in folders) | General tree |
| **HTML DOM** in every browser, JSON/XML | General tree |
| **Databases & file systems indexes** (MySQL InnoDB, PostgreSQL) | **B-trees / B+ trees** |
| Java `TreeMap`, C++ `std::map`, Linux CPU scheduler (CFS) | **Red-Black trees** |
| **Compilers & Python itself** | Abstract Syntax Tree (`import ast`) |
| **Machine learning** | **Decision trees**, Random Forest, XGBoost |
| Company org charts, comment threads, categories | General tree |
| Compression (ZIP) | Huffman tree (see 02.5) |
| Autocomplete | Trie (see 01.8) |

In [1]:
# Python itself parses your code into a tree:
import ast
print(ast.dump(ast.parse("x = 1 + 2 * 3").body[0].value, indent=2))

BinOp(
  left=Constant(value=1),
  op=Add(),
  right=BinOp(
    left=Constant(value=2),
    op=Mult(),
    right=Constant(value=3)))


## 1. Node class + helpers (same `TreeNode` LeetCode uses)

In [2]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values):
    """LeetCode-style level-order list -> tree. e.g. [3,9,20,None,None,15,7]"""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def tree_to_list(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node:
            out.append(node.val); q.append(node.left); q.append(node.right)
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

## 2. Traversals - the most important tree skill
| Traversal | Order | Use |
|---|---|---|
| **Pre-order** | node, left, right | Copy / serialise a tree |
| **In-order** | left, node, right | **BST → sorted order** |
| **Post-order** | left, right, node | Delete tree, compute sizes/heights bottom-up |
| **Level-order (BFS)** | level by level | Shortest depth, views, printing by level |

In [3]:
def preorder(node):
    return [node.val] + preorder(node.left) + preorder(node.right) if node else []

def inorder(node):
    return inorder(node.left) + [node.val] + inorder(node.right) if node else []

def postorder(node):
    return postorder(node.left) + postorder(node.right) + [node.val] if node else []

def inorder_iterative(root):          # interviewers like asking for the iterative version
    result, stack, cur = [], [], root
    while cur or stack:
        while cur:                    # go left as far as possible
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        result.append(cur.val)
        cur = cur.right
    return result

def level_order(root):
    if not root:
        return []
    result, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        result.append(level)
    return result

root = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7])
print("pre :", preorder(root))
print("in  :", inorder(root), "<- sorted, because it's a BST")
print("post:", postorder(root))
print("lvl :", level_order(root))
assert inorder(root) == inorder_iterative(root)

pre : [8, 3, 1, 6, 4, 7, 10, 14]
in  : [1, 3, 4, 6, 7, 8, 10, 14] <- sorted, because it's a BST
post: [1, 4, 7, 6, 3, 14, 10, 8]
lvl : [[8], [3, 10], [1, 6, 14], [4, 7]]


## 3. Build a Binary Search Tree from scratch

In [4]:
class BST:
    def __init__(self):
        self.root = None

    def insert(self, val):                       # O(h)
        def _insert(node, val):
            if not node:
                return TreeNode(val)
            if val < node.val:
                node.left = _insert(node.left, val)
            elif val > node.val:
                node.right = _insert(node.right, val)
            return node                          # duplicates ignored
        self.root = _insert(self.root, val)

    def search(self, val):                       # O(h)
        node = self.root
        while node:
            if val == node.val:
                return True
            node = node.left if val < node.val else node.right
        return False

    def delete(self, val):                       # O(h)
        def _min(node):
            while node.left:
                node = node.left
            return node
        def _delete(node, val):
            if not node:
                return None
            if val < node.val:
                node.left = _delete(node.left, val)
            elif val > node.val:
                node.right = _delete(node.right, val)
            else:
                if not node.left:                # 0 or 1 child
                    return node.right
                if not node.right:
                    return node.left
                successor = _min(node.right)     # 2 children: replace with in-order successor
                node.val = successor.val
                node.right = _delete(node.right, successor.val)
            return node
        self.root = _delete(self.root, val)

    def sorted_values(self):
        return inorder(self.root)

bst = BST()
for x in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    bst.insert(x)
print(bst.sorted_values())
assert bst.search(6) and not bst.search(5)
bst.delete(3)      # node with two children
bst.delete(14)     # node with one child
bst.delete(1)      # leaf
print(bst.sorted_values())
assert bst.sorted_values() == [4, 6, 7, 8, 10, 13]
print("BST ")

[1, 3, 4, 6, 7, 8, 10, 13, 14]
[4, 6, 7, 8, 10, 13]
BST 


---
## Solved problems

### 1. Symmetric Tree - LC #101 (Easy)
Is the tree a mirror of itself?

**Idea:** Compare left subtree with right subtree mirrored: outer pair and inner pair.

In [5]:
def is_symmetric(root):
    def mirror(a, b):
        if not a and not b:
            return True
        if not a or not b or a.val != b.val:
            return False
        return mirror(a.left, b.right) and mirror(a.right, b.left)
    return mirror(root.left, root.right) if root else True

assert is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3]))
assert not is_symmetric(build_tree([1, 2, 2, None, 3, None, 3]))
print("Symmetric ")

Symmetric 


### 2. Path Sum - LC #112 (Easy)
Is there a root-to-leaf path whose values sum to `target`?

In [6]:
def has_path_sum(root, target):
    if not root:
        return False
    if not root.left and not root.right:          # leaf
        return root.val == target
    remaining = target - root.val
    return has_path_sum(root.left, remaining) or has_path_sum(root.right, remaining)

t = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])
assert has_path_sum(t, 22) and has_path_sum(t, 27) and not has_path_sum(t, 100)   # 5-4-11-2, 5-4-11-7
assert not has_path_sum(None, 0)
print("Path Sum ")

Path Sum 


### 3. Diameter of Binary Tree - LC #543 (Easy) Length (in edges) of the longest path between any two nodes.

**Idea:** Post-order: each node returns its height; the longest path through a node = left height + right height. Track the best globally.

In [7]:
def diameter_of_binary_tree(root):
    best = 0
    def height(node):
        nonlocal best
        if not node:
            return 0
        left, right = height(node.left), height(node.right)
        best = max(best, left + right)       # path passing through this node
        return 1 + max(left, right)
    height(root)
    return best

assert diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])) == 3
assert diameter_of_binary_tree(build_tree([1, 2])) == 1
print("Diameter ")

Diameter 


### 4. Balanced Binary Tree - LC #110 (Easy)
Height-balanced = for every node, left and right heights differ by ≤ 1.

In [8]:
def is_balanced(root):
    def check(node):              # returns height, or -1 if unbalanced
        if not node:
            return 0
        left = check(node.left)
        right = check(node.right)
        if left == -1 or right == -1 or abs(left - right) > 1:
            return -1
        return 1 + max(left, right)
    return check(root) != -1

assert is_balanced(build_tree([3, 9, 20, None, None, 15, 7]))
assert not is_balanced(build_tree([1, 2, 2, 3, 3, None, None, 4, 4]))
print("Balanced ")

Balanced 


### 5. Binary Tree Right Side View - LC #199 (Medium)
Values visible when looking at the tree from the right.

**Idea:** BFS level by level; take the rightmost node of each level.

In [9]:
def right_side_view(root):
    if not root:
        return []
    view, q = [], deque([root])
    while q:
        for i in range(len(q)):
            node = q.popleft()
            if i == 0:
                view.append(node.val)          # first node of level (we push right first)
            if node.right: q.append(node.right)
            if node.left: q.append(node.left)
    return view

assert right_side_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
assert right_side_view(build_tree([1, 2])) == [1, 2]
print("Right Side View ")

Right Side View 


### 6. Kth Smallest Element in a BST - LC #230 (Medium)


**Idea:** In-order traversal of a BST visits values in sorted order - stop at the k-th one.

In [10]:
def kth_smallest(root, k):
    stack, cur = [], root
    while True:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        k -= 1
        if k == 0:
            return cur.val
        cur = cur.right

t = build_tree([5, 3, 6, 2, 4, None, None, 1])
assert [kth_smallest(t, k) for k in range(1, 7)] == [1, 2, 3, 4, 5, 6]
print("Kth Smallest ")

Kth Smallest 


### 7. Serialize and Deserialize Binary Tree - LC #297 (Hard) Convert a tree to a string and back - what you do to save it to a file, database or send over a network.

**Idea:** Pre-order DFS writing `#` for empty children; read it back in the same pre-order.

In [11]:
def serialize(root):
    out = []
    def dfs(node):
        if not node:
            out.append("#")
            return
        out.append(str(node.val))
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return ",".join(out)

def deserialize(data):
    values = iter(data.split(","))
    def build():
        v = next(values)
        if v == "#":
            return None
        node = TreeNode(int(v))
        node.left = build()
        node.right = build()
        return node
    return build()

t = build_tree([1, 2, 3, None, None, 4, 5])
s = serialize(t)
print(s)
assert tree_to_list(deserialize(s)) == [1, 2, 3, None, None, 4, 5]
assert deserialize(serialize(None)) is None
print("Serialize/Deserialize ")

1,2,#,#,3,4,#,#,5,#,#
Serialize/Deserialize 


## More practice on LeetCode
Easy: Same Tree (#100), Subtree of Another Tree (#572), Minimum Depth (#111), Convert Sorted Array to BST (#108)
Medium: Construct Tree from Preorder & Inorder (#105), Count Good Nodes (#1448), Lowest Common Ancestor (#236), Delete Node in a BST (#450)
Hard: Binary Tree Maximum Path Sum (#124)

Also in `03-Interview Patterns/03.1`: Max Depth, Invert Tree, Level Order, Validate BST, LCA of BST.

## Key takeaways
- Most tree problems = **DFS recursion**: base case `if not node`, solve children, combine.
- BFS with a `deque` for anything "per level".
- **In-order traversal of a BST = sorted order.**
- Complexity is usually O(n) time, O(h) space (recursion depth).

**Next: `01.6-Heaps-and-Priority-Queues.ipynb`**